# License Plate Detection (Vehicle + License Plate Detection + OCR)

Pipeline:
1. **YOLO** for vehicle detection and vehicle type (Car / Motorcycle / Bus / Truck / ...)
2. **YOLOv8 plate model** to find the plate inside each vehicle box
3. **OCR**: Persian/Arabic-script plates -> **Hezar CRNN**, Latin-script plates -> **fast-plate-ocr** (both fall back to EasyOCR)
4. Country-specific parsing via `COUNTRY_MODE` + lightweight tracking so the printed plate text doesn't flicker frame to frame

Country modes:
- `IR` -- full Iranian plate parsing: digits/letter, province code, colour, plate type, free-trade-zone
- `GLOBAL` -- detection + Latin text only, no country-specific parsing
- `AUTO` -- per plate: valid Iranian plate -> Iran parsing, otherwise -> Latin

Notebook structure:
- Section 1: setup & imports
- Section 2: configuration (paths, country mode, vehicle classes, parameters)
- Section 3: automatic GPU/CPU selection + model loading
- Section 4: helper functions (OCR engines, Persian text rendering on video)
- Section 5: Iran plate data & parsing logic (province codes, colours, categories)
- Section 6: main video-processing function (tracking + strongest reading per plate)
- Section 7: run on test videos + save/display output
- Section 8: build a GIF from the output

---

## Tuning accuracy/speed per camera (zero code changes)

The detection and OCR logic in this notebook is fixed; **only two parameters** are read from a config file, and a third is derived from them automatically -- so you can retune the pipeline for a different camera or scene without touching a single line of code:

- **Vehicle model** (manual): which YOLO checkpoint to use for vehicle detection (lighter/faster vs. heavier/more accurate)
- **Vehicle `imgsz`** (manual): the input resolution -- this is given **only** to the vehicle detector, never to the plate detector
- **`OCR_MIN_H`** (automatic, derived) = `64 x imgsz/640`: a larger `imgsz` means farther/smaller vehicles become visible whose plates are genuinely tiny in real pixels; this parameter scales up the *pre-OCR* upscaling threshold so those small plates still get a fair shot at OCR.

These parameters are read from `PelakX/configs/pelak3.yaml`. Four ready-made profiles ship in `PelakX/configs/`:

| File | imgsz | OCR_MIN_H (auto) | Best for |
|---|---|---|---|
| `baseline_640.yaml` | 640 | 64 | this notebook's baseline behaviour (for regression testing) |
| `pelak3.yaml` (default) | 960 | 96 | balanced -- good for light traffic, acceptable on highways |
| `highway.yaml` | 1280 | 128 | highway |
| `highway_far.yaml` | 1600 | 160 | highway / very far cameras |

To switch profiles, point `CONFIG_PATH` (in the settings cell) at one of these files. If the config file is missing, the notebook falls back to baseline behaviour (640).

### A couple of engineering lessons baked into these defaults

1. **Only give `imgsz` to the vehicle detector, never the plate detector.** The plate model is tuned for its default input size; changing it shifts the plate bounding box slightly, and that small shift goes straight into OCR accuracy. The right lever for "smart" cropping is upscaling *after* the plate is found (`OCR_MIN_H`), not changing detection itself.
2. **Process every frame, don't skip frames.** Skipping frames to save compute makes the plate box/label disappear every other frame and the video flicker -- the UX cost is worse than the compute it saves.

**Note:** no OCR rate-limiting (e.g. "at most N reads per second per track") is implemented here -- everything is simple and direct.


## Section 1 -- Setup & imports

In [ ]:
# Not needed if you're using the "Python (pelak)" kernel (comment left as reference).
# %pip install -q opencv-python ultralytics easyocr numpy torch torchvision
# %pip install -q hezar arabic-reshaper python-bidi      # Persian OCR + RTL text rendering on video
# %pip install -q fast-plate-ocr onnxruntime             # Accurate Latin-script plate OCR (65+ countries)

In [ ]:
import csv
import urllib.request
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO
import easyocr

## Section 2 -- Configuration

Every parameter and path lives here, and only here.

In [ ]:
# --- Paths (absolute) ---
PROJECT_DIR = Path(r"C:\1\1_پروژه\تشخیص پلاکو ماشین")
VIDEO_DIR = PROJECT_DIR / "ویدیو"      # input videos folder (kept as the real folder name on disk)
OUTPUT_DIR = PROJECT_DIR / "خروجی"     # outputs are written here (kept as the real folder name on disk)
MODELS_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# ============================================================================
#  Per-camera config  --  PelakX/configs/pelak3.yaml
# ============================================================================
#  Only two parameters are read from a YAML file so a camera/video can be
#  retuned with zero code changes:
#    - vehicle_model : the YOLO checkpoint used for vehicle detection
#    - imgsz          : input resolution for *only* the vehicle model
#  imgsz is deliberately NOT applied to the plate detector -- testing showed
#  that changing the plate model's input size shifts its bounding box
#  slightly, and that small shift goes straight into OCR accuracy.
#
#  Smart scaling: a larger vehicle imgsz means farther/smaller vehicles
#  become visible, whose plates are genuinely tiny in real pixels. So the
#  pre-OCR upscale threshold (OCR_MIN_H, below) is derived automatically
#  from IMGSZ -- not by touching the plate detector, but by upscaling the
#  final crop more *after* the plate has already been found.
#  If the config file is missing, baseline behaviour (imgsz=640) is used.
# ============================================================================
import yaml

CONFIG_PATH = PROJECT_DIR / "PelakX" / "configs" / "pelak3.yaml"  # point this at another profile for a different camera/video
_cfg = {}
if CONFIG_PATH.exists():
    _cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    print(f"⚙️ Config loaded: {CONFIG_PATH.name}")
else:
    print(f"⚠️ Config file not found ({CONFIG_PATH}) -- falling back to baseline behaviour (imgsz=640).")

# Vehicle model: whatever the config says (default yolo26s.pt); if not present
# locally, the name is handed to ultralytics, which downloads it itself.
_wanted_model = _cfg.get("vehicle_model", "yolo26s.pt")
_local_wanted = PROJECT_DIR / _wanted_model
VEHICLE_MODEL_PATH = str(_local_wanted) if _local_wanted.exists() else _wanted_model

PLATE_MODEL_PATH = MODELS_DIR / "license_plate_model.pt"
PLATE_MODEL_URL = (
    "https://github.com/Muhammad-Zeerak-Khan/"
    "Automatic-License-Plate-Recognition-using-YOLOv8/raw/main/license_plate_detector.pt"
)

# ============================================================================
#  Country mode  --  COUNTRY_MODE  (one of: "IR" | "GLOBAL" | "AUTO")
# ============================================================================
#   "IR"     -> full Iranian plate parsing: digits/letter, province code, colour, plate type, free-trade-zone
#   "GLOBAL" -> plate detection + Latin text only, no country-specific parsing
#   "AUTO"   -> per plate: valid Iranian plate -> Iran parsing, otherwise -> Latin
#
# The full 12-country grammar engine lives in the separate PelakX package
# (PelakX/configs/countries/). This notebook only natively parses IR; every
# other country is read as plain Latin text.
# ============================================================================
SUPPORTED_COUNTRIES = {
    "IR": "Iran 🇮🇷 (fully parsed in this notebook)",
    "AE": "UAE 🇦🇪", "BR": "Brazil 🇧🇷", "DE": "Germany 🇩🇪", "ES": "Spain 🇪🇸",
    "FR": "France 🇫🇷", "GB": "United Kingdom 🇬🇧", "IN": "India 🇮🇳", "IT": "Italy 🇮🇹",
    "NL": "Netherlands 🇳🇱", "TR": "Türkiye 🇹🇷", "US": "United States 🇺🇸",
}

COUNTRY_MODE = "AUTO"

# --- Vehicle classes (COCO classes the vehicle model detects) ---
VEHICLE_NAMES = {
    1: "Bicycle", 2: "Car", 3: "Motorcycle", 5: "Bus", 6: "Train", 7: "Truck",
}
VEHICLE_CLASSES = list(VEHICLE_NAMES)   # only look for/label plates on these classes

IR_LOGIC = COUNTRY_MODE in ("IR", "AUTO")
OCR_LANGS = ["fa", "en"] if IR_LOGIC else ["en"]

# --- Detection thresholds ---
VEHICLE_CONF = 0.5
PLATE_CONF = 0.4
PROGRESS_EVERY = 30

# Input resolution for *only* the vehicle model (imgsz). Larger = better
# accuracy on small/distant vehicles (highways), at a higher compute cost.
# The plate detector is not affected.
#   640  -> baseline behaviour / nearby streets
#   960  -> balanced
#   1280 -> highway
#   1600 -> highway / very far cameras
IMGSZ = int(_cfg.get("imgsz", 960))

# Pre-OCR upscale threshold, automatic and proportional to IMGSZ. At
# imgsz=640 this is exactly 64 (baseline behaviour); at larger imgsz values,
# small plates on distant vehicles get proportionally more upscaling so OCR
# still gets a fair-sized image to work with.
#   640->64 | 960->96 | 1280->128 | 1600->160
OCR_MIN_H = round(64 * IMGSZ / 640)

print("Available country modes:  IR  |  GLOBAL  |  AUTO")
for _c, _n in SUPPORTED_COUNTRIES.items():
    print(f"   {_c} -- {_n}")
print(f"\nDetectable vehicle types: {', '.join(VEHICLE_NAMES.values())}")
print(f"✅ Selected mode: {COUNTRY_MODE}   |   Project dir: {PROJECT_DIR}")
print(f"✅ Vehicle model: {Path(VEHICLE_MODEL_PATH).name}  |  imgsz (vehicle only)={IMGSZ}"
      f"  |  OCR_MIN_H (auto)={OCR_MIN_H}"
      f"  |  vehicle_conf={VEHICLE_CONF}  |  plate_conf={PLATE_CONF}")


## Section 3 -- Automatic GPU/CPU selection & model loading

In [ ]:
# Automatic selection: CUDA GPU if available, else CPU
USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else "cpu"
print(f"🖥️ Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if USE_GPU else 'CPU'}")

In [ ]:
# Download the plate model (first run only)
if not PLATE_MODEL_PATH.exists():
    print("📥 Downloading the plate model from GitHub...")
    req = urllib.request.Request(PLATE_MODEL_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(PLATE_MODEL_PATH, "wb") as f:
        f.write(resp.read())
    print(f"✅ Saved: {PLATE_MODEL_PATH}")
else:
    print(f"✅ Plate model already present: {PLATE_MODEL_PATH}")

In [ ]:
vehicle_model = YOLO(VEHICLE_MODEL_PATH)
plate_model = YOLO(str(PLATE_MODEL_PATH))

# --- Persian OCR: Hezar CRNN (falls back to EasyOCR if it fails to load) ---
easy_reader = easyocr.Reader(OCR_LANGS, gpu=USE_GPU)
hezar_model = None
if IR_LOGIC:
    try:
        from hezar.models import Model
        hezar_model = Model.load("hezarai/crnn-fa-license-plate-recognition-v2")
        print("✅ Persian OCR: Hezar CRNN")
    except Exception as e:
        print(f"⚠️ Hezar failed to load ({e}) -- fallback: EasyOCR")

# --- Latin OCR: fast-plate-ocr (global model, very accurate on Latin-script plates) ---
latin_ocr = None
if COUNTRY_MODE in ("GLOBAL", "AUTO"):
    try:
        from fast_plate_ocr import LicensePlateRecognizer
        # cct-s = more accurate and a bit heavier; cct-xs = lighter
        latin_ocr = LicensePlateRecognizer("cct-s-v2-global-model")
        print("✅ Latin OCR: fast-plate-ocr (cct-s-v2-global)")
    except Exception as e:
        print(f"⚠️ fast-plate-ocr failed to load ({e}) -- fallback: EasyOCR")

print(f"✅ Vehicle model: {Path(VEHICLE_MODEL_PATH).name}  |  Mode: {COUNTRY_MODE}")

## Section 4 -- Helper functions

In [ ]:
from PIL import Image, ImageDraw, ImageFont

try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    _HAS_RTL = True
except Exception:
    _HAS_RTL = False

# A font with Persian glyphs (Tahoma ships on every Windows install)
try:
    _FONT = ImageFont.truetype(r"C:\Windows\Fonts\tahoma.ttf", 22)
except Exception:
    _FONT = ImageFont.load_default()

_DIGITS_TO_ASCII = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_is_fa = lambda s: any("؀" <= c <= "ۿ" for c in s)


def _shape_fa(text):
    """Shape Persian text for correct rendering (letter joining + right-to-left)."""
    if _HAS_RTL and _is_fa(text):
        try:
            return get_display(arabic_reshaper.reshape(text))
        except Exception:
            return text
    return text


def draw_box(frame, x1, y1, x2, y2, color):
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)


def draw_plate_label(frame, x1, y1, text, color_bgr):
    """Draws the label above a plate with Pillow, so Persian text/digits render correctly."""
    if not text:
        return
    disp = _shape_fa(str(text))
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(img)
    l, t, r, b = d.textbbox((0, 0), disp, font=_FONT)
    tw, th = r - l, b - t
    ty = max(0, y1 - th - 10)
    fill = (color_bgr[2], color_bgr[1], color_bgr[0])
    d.rectangle([x1, ty, x1 + tw + 12, ty + th + 10], fill=fill)
    d.text((x1 + 6, ty + 4), disp, font=_FONT, fill=(0, 0, 0))
    frame[:] = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def _upscale(crop, min_h=OCR_MIN_H):
    """Upscales small plate crops -- significantly improves CRNN/OCR accuracy.

    The default min_h comes from OCR_MIN_H: at imgsz=640 (baseline) it is
    exactly 64; at larger imgsz values (farther/smaller vehicles become
    visible) it grows automatically so the small plate crop gets upscaled
    enough before OCR.
    """
    h, w = crop.shape[:2]
    if 0 < h < min_h:
        f = min_h / h
        crop = cv2.resize(crop, (int(w * f), min_h), interpolation=cv2.INTER_CUBIC)
    return crop


# ---------- OCR engines ----------
def _ocr_easy(crop):
    res = easy_reader.readtext(_upscale(crop))
    if not res:
        return "", 0.0
    res.sort(key=lambda r: r[0][0][0])
    return " ".join(r[1] for r in res).strip(), float(np.mean([r[2] for r in res]))


def _ocr_hezar(crop):
    out = hezar_model.predict(_upscale(crop))
    while isinstance(out, (list, tuple)) and out:
        out = out[0]
    if isinstance(out, dict):
        return str(out.get("text", "")).strip(), float(out.get("score", 0.9) or 0.9)
    if hasattr(out, "text"):
        return str(out.text).strip(), float(getattr(out, "score", 0.9) or 0.9)
    return (str(out).strip() if out is not None else ""), 0.9


def _ocr_fastplate(crop):
    crop = _upscale(crop)
    if crop.ndim == 2:
        crop = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
    try:
        out = latin_ocr.run(crop, return_confidence=True)
    except TypeError:
        out = latin_ocr.run(crop)
    if isinstance(out, tuple) and len(out) >= 2:
        out = out[0]
    item = out[0] if isinstance(out, (list, tuple)) and out else out
    text = str(getattr(item, "plate", item if isinstance(item, str) else "") or "").replace("_", "").strip()
    probs = getattr(item, "char_probs", None)
    arr = np.asarray(probs, float).reshape(-1) if probs is not None else np.array([])
    conf = float(arr.mean()) if arr.size else 0.9
    return text, max(0.0, min(1.0, conf))


def _read_fa(crop):
    return _ocr_hezar(crop) if hezar_model is not None else _ocr_easy(crop)


def _read_latin(crop):
    return _ocr_fastplate(crop) if latin_ocr is not None else _ocr_easy(crop)


def read_plate(crop):
    """Plate image -> (text, confidence, script).  script: 'fa' or 'latin'.

    AUTO: read with the Persian engine first; only keep that result if it is
    a *valid* Iranian plate, otherwise fall back to the accurate Latin
    engine (the Persian engine always returns *something* in Persian script,
    even for a foreign plate).
    """
    if crop is None or crop.size == 0:
        return "", 0.0, "latin"
    try:
        if COUNTRY_MODE == "IR":
            t, c = _read_fa(crop)
            return t, c, "fa"
        if COUNTRY_MODE == "GLOBAL":
            t, c = _read_latin(crop)
            return t, c, "latin"
        # --- AUTO ---
        t_fa, c_fa = _read_fa(crop)
        if parse_iranian_plate(t_fa, crop)["valid"]:
            return t_fa, c_fa, "fa"
        t_lat, c_lat = _read_latin(crop)
        if t_lat:
            return t_lat, c_lat, "latin"
        return t_fa, c_fa, "fa"
    except Exception:
        t, c = _ocr_easy(crop)
        return t, c, ("fa" if _is_fa(t) else "latin")

## Section 5 -- Iran plate data & parsing logic

Source: `PelakX/configs/countries/ir.yaml` (province codes from ghabzino + Wikipedia, letter semantics
from photographed real-world samples). Runs on every plate in `IR` mode; only on Persian-script plates
in `AUTO` mode.

> **Note:** the field values below (province names, plate-type labels, colour names) are intentionally
> kept in Persian -- they are the real, official Iranian terms, not UI language. Only the surrounding
> comments here are in English.


In [ ]:
# Two-digit province codes (shared codes match the real system exactly).
# NOTE: kept in Persian on purpose -- these are the real official province names,
# used as actual output field values, not UI text.
PROVINCE_CODES = {
    "10": "تهران", "11": "تهران", "12": "خراسان رضوی", "13": "اصفهان", "14": "خوزستان",
    "15": "آذربایجان شرقی", "16": "قم", "17": "آذربایجان غربی", "18": "همدان", "19": "کرمانشاه",
    "20": "تهران", "21": "البرز", "22": "تهران", "23": "اصفهان", "24": "خوزستان",
    "25": "آذربایجان شرقی", "26": "خراسان شمالی", "27": "آذربایجان غربی", "28": "همدان",
    "29": "کرمانشاه", "30": "تهران/البرز", "31": "لرستان", "32": "خراسان رضوی/شمالی/جنوبی",
    "33": "تهران", "34": "خوزستان", "35": "آذربایجان شرقی", "36": "خراسان رضوی",
    "37": "آذربایجان غربی", "38": "البرز", "39": "کرمانشاه", "40": "تهران", "41": "لرستان",
    "42": "خراسان رضوی", "43": "اصفهان", "44": "تهران", "45": "کرمان", "46": "گیلان",
    "47": "مرکزی", "48": "بوشهر", "49": "کهگیلویه و بویراحمد", "50": "تهران", "51": "کردستان",
    "52": "خراسان جنوبی", "53": "اصفهان", "54": "یزد", "55": "تهران", "56": "گیلان",
    "57": "مرکزی", "58": "بوشهر", "59": "گلستان", "60": "تهران", "61": "کردستان",
    "62": "مازندران", "63": "فارس", "64": "یزد", "65": "کرمان", "66": "تهران",
    "67": "اصفهان", "68": "البرز", "69": "گلستان", "71": "چهارمحال و بختیاری",
    "72": "مازندران", "73": "فارس", "74": "خراسان رضوی/شمالی", "75": "کرمان",
    "76": "گیلان", "77": "تهران", "78": "تهران/البرز", "79": "قزوین",
    "81": "چهارمحال و بختیاری", "82": "مازندران", "83": "فارس", "84": "هرمزگان",
    "85": "سیستان و بلوچستان", "86": "سمنان", "87": "زنجان", "88": "تهران", "89": "قزوین",
    "91": "اردبیل", "92": "مازندران", "93": "فارس", "94": "هرمزگان", "95": "سیستان و بلوچستان",
    "96": "سمنان", "97": "زنجان", "98": "ایلام", "99": "تهران",
}

# Plate letter -> meaning (plate type) + expected background colour.
# NOTE: kept in Persian on purpose -- see note above.
LETTER_SEMANTICS = {
    "الف": ("دولتی", "قرمز"), "پ": ("پلیس/انتظامی", "سبز"),
    "ت": ("تاکسی", "زرد"), "ع": ("حمل‌ونقل عمومی", "زرد"),
    "ک": ("ماشین‌آلات کشاورزی", "زرد"), "ژ": ("جانبازان و معلولین", "سفید"),
    "گ": ("گذر موقت", "سفید"), "ث": ("سپاه (تأیید‌نشده)", "?"),
    "D": ("دیپلمات", "آبی"), "S": ("سیاسی", "آبی"),
    "معلولین": ("جانبازان و معلولین", "سفید"),
    "تشریفات": ("تشریفات", "قرمز"), "موقت": ("موقت منطقه آزاد", "?"),
}

# Letters allowed in the letter slot of civilian/public plates
IR_LETTERS = list("بپتثجحدزژسشصطعفقکگلمنوهی") + ["الف", "D", "S"]

FA_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
NOISE_WORDS = ["ایران", "IRAN", "IR", "منطقه آزاد", "آزاد"]
print("✅ Iran plate reference data loaded.")

In [ ]:
def detect_plate_color(plate_crop):
    """Returns the plate's dominant background colour: white/yellow/red/green/blue/unknown."""
    if plate_crop is None or plate_crop.size == 0:
        return "نامشخص"
    hsv = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2HSV)
    h, s, v = (int(np.median(hsv[:, :, i])) for i in range(3))
    if s < 60 and v > 120:
        return "سفید"
    if s < 60:
        return "نامشخص"
    if h < 12 or h > 168:
        return "قرمز"
    if 15 <= h <= 38:
        return "زرد"
    if 40 <= h <= 85:
        return "سبز"
    if 90 <= h <= 140:
        return "آبی"
    return "نامشخص"


def parse_iranian_plate(raw_text, plate_crop):
    """Raw OCR text + plate image -> parsed dict."""
    t = raw_text.translate(_DIGITS_TO_ASCII)
    for w in NOISE_WORDS:
        t = t.replace(w, " ")
    digits = "".join(c for c in t if c.isdigit())
    letters = (["الف"] if "الف" in t else []) + [c for c in t if c in IR_LETTERS]
    letter = letters[0] if letters else ""

    color = detect_plate_color(plate_crop)
    plate_type, _ = LETTER_SEMANTICS.get(letter, ("شخصی", "سفید"))
    if not letter and color == "زرد":
        plate_type = "عمومی/تاکسی (از روی رنگ)"

    out = {
        "raw": raw_text, "digits": digits, "letter": letter,
        "color": color, "type": plate_type, "province": "", "province_code": "",
        "free_zone": ("منطقه آزاد" in raw_text or "موقت" in raw_text),
        "formatted": raw_text, "valid": False,
    }

    if len(digits) >= 7 and letter:          # civilian/public layout: DD L DDD + province code DD
        left, right, prov = digits[:2], digits[2:5], digits[5:7]
        out["province_code"] = prov
        out["province"] = PROVINCE_CODES.get(prov, "نامشخص")
        out["formatted"] = f"{left} {letter} {right} - ایران {prov}"
        out["valid"] = True
    elif len(digits) == 8 and not letter:    # motorcycle
        out["type"] = "موتورسیکلت"
        out["province_code"] = digits[:3]
        out["formatted"] = f"{digits[:3]} - {digits[3:]}"
        out["valid"] = True

    return out


def ir_video_label(info):
    """On-video label for an Iranian plate (rendered with Pillow, so Persian text is fine)."""
    if info["valid"]:
        return info["formatted"]
    return info["digits"] or info["raw"] or "PLATE"


def latin_video_label(text):
    return text or "PLATE"

## Section 6 -- Main video-processing function

In [ ]:
CSV_HEADER = ["track", "vehicle", "script", "raw", "digits", "letter", "color",
              "type", "province", "formatted", "conf", "frames_seen"]

MATCH_DIST = 120     # max centre-to-centre distance (px) between frames to count as "the same plate"
MIN_FRAMES = 2       # a plate seen fewer times than this is treated as noise


def process_video(video_path, output_path, csv_path=None, max_frames=None):
    """Input video -> annotated video + CSV with one row per plate (strongest reading).

    - Every vehicle is labelled with its type (Car / Bus / Truck / Motorcycle / ...).
    - Each plate is a "track"; the printed text only changes when a higher-confidence
      reading comes in, so it doesn't flicker on the video.
    - Every frame is processed -- no frame is skipped.
    max_frames: quick-test only; None means the whole video.
    """
    video_path, output_path = str(video_path), str(output_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    tracks = []
    print(f"🚀 Processing {Path(video_path).name} -- {total} frames (mode {COUNTRY_MODE}, vehicle imgsz={IMGSZ})")
    frame_i = 0
    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frame_i >= max_frames):
            break
        frame_i += 1
        if frame_i % PROGRESS_EVERY == 0:
            print(f"⏳ Frame {frame_i}/{total}")

        # imgsz is applied only to the vehicle model; the plate model is
        # deliberately left untouched (no imgsz, default size) so the OCR
        # path stays as accurate as possible.
        v_res = vehicle_model.predict(frame, imgsz=IMGSZ, verbose=False, device=DEVICE)[0]
        for box in v_res.boxes:
            cls, conf = int(box.cls[0]), float(box.conf[0])
            if cls not in VEHICLE_CLASSES or conf < VEHICLE_CONF:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            vname = VEHICLE_NAMES.get(cls, "Vehicle")
            draw_box(frame, x1, y1, x2, y2, (255, 150, 0))
            draw_plate_label(frame, x1, y1, f"{vname} {conf:.2f}", (255, 150, 0))

            v_crop = frame[y1:y2, x1:x2]
            if v_crop.size == 0 or cls in (1, 6):   # bicycles/trains don't have plates
                continue

            p_res = plate_model.predict(v_crop, verbose=False, device=DEVICE)[0]
            for p in p_res.boxes:
                if float(p.conf[0]) < PLATE_CONF:
                    continue
                px1, py1, px2, py2 = map(int, p.xyxy[0])
                gx1, gy1, gx2, gy2 = x1 + px1, y1 + py1, x1 + px2, y1 + py2
                cx, cy = (gx1 + gx2) // 2, (gy1 + gy2) // 2
                plate_img = v_crop[py1:py2, px1:px2]

                text, ocr_conf, script = read_plate(plate_img)

                tr, dmin = None, 1e9
                for t in tracks:
                    d = ((t["cx"] - cx) ** 2 + (t["cy"] - cy) ** 2) ** 0.5
                    if d < dmin:
                        dmin, tr = d, t
                if tr is None or dmin > MATCH_DIST:
                    tr = {"cx": cx, "cy": cy, "vehicle": vname, "text": "", "conf": -1.0,
                          "script": script, "info": None, "color": "", "frames": 0}
                    tracks.append(tr)
                tr["cx"], tr["cy"], tr["vehicle"], tr["frames"] = cx, cy, vname, tr["frames"] + 1

                # Strongest reading wins: only replace when confidence is higher
                if text and ocr_conf > tr["conf"]:
                    tr["text"], tr["conf"], tr["script"] = text, ocr_conf, script
                    tr["color"] = detect_plate_color(plate_img)
                    tr["info"] = parse_iranian_plate(text, plate_img) if script == "fa" else None

                draw_box(frame, gx1, gy1, gx2, gy2, (0, 255, 0))
                if tr["script"] == "fa" and tr["info"]:
                    label = ir_video_label(tr["info"])
                else:
                    label = latin_video_label(tr["text"])
                draw_plate_label(frame, gx1, gy1, label, (0, 255, 0))

        out.write(frame)

    cap.release()
    out.release()

    # --- CSV: one row per plate (strongest reading) ---
    rows = []
    for i, tr in enumerate(tracks):
        if not tr["text"] or tr["frames"] < MIN_FRAMES:
            continue
        info = tr["info"]
        if tr["script"] == "fa" and info:
            rows.append([i, tr["vehicle"], "fa", info["raw"], info["digits"], info["letter"],
                         info["color"], info["type"], info["province"],
                         info["formatted"], round(tr["conf"], 3), tr["frames"]])
        else:
            rows.append([i, tr["vehicle"], tr["script"], tr["text"], "", "", tr["color"],
                         "", "", tr["text"], round(tr["conf"], 3), tr["frames"]])

    if csv_path:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(CSV_HEADER)
            w.writerows(rows)
        print(f"📄 CSV: {Path(csv_path).name}  ({len(rows)} unique plates)")

    print(f"✅ Output video: {output_path}")
    return output_path, rows


## Section 7 -- Run on test videos

Put your test video path(s) in `TEST_VIDEOS` (a full path or a filename inside the `ویدیو` folder).
For every video, a `<name>_out.mp4` and a `<name>_plates.csv` are written to the `خروجی` folder.

In [ ]:
TEST_VIDEOS = [
    VIDEO_DIR / "1_P.mp4",
    # VIDEO_DIR / "2-1_P.mp4",
    # VIDEO_DIR / "2-2_P.mp4",
]

results = []
for video in TEST_VIDEOS:
    video = Path(video)
    if not video.exists():
        print(f"⚠️ Not found: {video}")
        continue
    results.append(process_video(
        video,
        OUTPUT_DIR / f"{video.stem}_out.mp4",
        OUTPUT_DIR / f"{video.stem}_plates.csv",
        max_frames=600,   # quick-test only; None for the whole video
    ))


In [ ]:
# Per-video summary of unique plates (strongest reading per plate)
import pandas as pd

for out_path, rows in results:
    print(f"\n🎥 {Path(out_path).name}  --  {len(rows)} unique plates")
    if not rows:
        continue
    df = pd.DataFrame(rows, columns=CSV_HEADER).sort_values("frames_seen", ascending=False)
    display(df[["vehicle", "script", "formatted", "type", "province", "color", "conf", "frames_seen"]])

In [ ]:
# Show the output video inline in the notebook
# (if your browser can't play the mp4v codec, open the file directly from the output folder)
from IPython.display import Video, display

for out_path, _ in results:
    display(Video(str(out_path), embed=True, width=640))

## Section 8 -- Build a GIF from the output

Makes a short GIF from each output video (for sharing). Saved to the `خروجی` folder.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image as IPyImage


def video_to_gif(mp4_path, gif_path, out_fps=8, scale=0.5, max_seconds=6):
    """Video -> a short, lightweight GIF."""
    cap = cv2.VideoCapture(str(mp4_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, round(src_fps / out_fps))
    max_frames = int(src_fps * max_seconds)

    frames, i = [], 0
    while True:
        ret, frame = cap.read()
        if not ret or i > max_frames:
            break
        if i % step == 0:
            if scale != 1:
                frame = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()

    imageio.mimsave(str(gif_path), frames, fps=out_fps, loop=0)
    mb = Path(gif_path).stat().st_size / 1e6
    print(f"🎞️ {Path(gif_path).name}  --  {len(frames)} frames, {mb:.1f} MB")
    return gif_path


for out_path, _ in results:
    gif = Path(out_path).with_suffix("").with_name(Path(out_path).stem + ".gif")
    video_to_gif(out_path, gif)
    display(IPyImage(filename=str(gif)))